In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pickle
import swifter

In [ ]:
# path = '../data/us_10m_nointernship_ai_skills_body.parquet.gzip'
path = '../data/us_10m_nointernship_2018_2024_benefits.parquet.gzip'

In [ ]:
if path[-3:] == 'csv:':
    usdf = pd.read_csv(path)
elif path[-3:] == 'zip':
    usdf = pd.read_parquet(path)

In [ ]:
# usdf.rename(columns = {'Has AI Skills': 'AI ROLE'}, inplace = True)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
usdf

# Experience

In [ ]:
# Define buckets for years of experience
bins = [-2, -1, 0, 2, 5, 10, 20, 100]
labels = ['Missing', '0 years', '1-2 years', '3-5 years', '6-10 years', '11-20 years', '21+ years']


In [ ]:
usdf['EXPERIENCE_BUCKET'] = pd.cut(usdf['MIN_YEARS_EXPERIENCE'], bins=bins, labels=labels, right=True)

In [ ]:
usdf['EXPERIENCE_BUCKET']=usdf['EXPERIENCE_BUCKET'].astype(str)

In [ ]:
# replace nan with 'None Listed'
usdf['EXPERIENCE_BUCKET'] = usdf['EXPERIENCE_BUCKET'].replace('nan', 'None Listed')

In [ ]:
usdf['EXPERIENCE_BUCKET'].value_counts(dropna=False)

In [ ]:
usdf.head()

In [ ]:
if path[-3:] == 'csv:':
    usdf.to_csv(path)
elif path[-3:] == 'zip':
    usdf.to_parquet(path, compression='gzip')

# Log Salary

In [ ]:
usdf['LOG_SALARY'] = np.log(usdf['SALARY'])

# Salary

In [ ]:
salaries = pd.read_csv('../data/SALARIES.csv')

In [ ]:
all_data = usdf.merge(salaries, left_on='ID', right_on='ID', how='left')

In [ ]:
usdf = all_data

In [ ]:
all_data.columns

In [ ]:
usdf.columns

In [ ]:
usdf['SALARY']

In [ ]:
usdf[usdf['SALARY']>=0]

In [ ]:
usdf['SALARY'].hist()

In [ ]:
usdf['LOG_SALARY'] = np.log(usdf['SALARY'])

In [ ]:
usdf['LOG_SALARY'].hist()

In [ ]:
if path[-3:] == 'csv:':
    usdf.to_csv(path)
elif path[-3:] == 'zip':
    usdf.to_parquet(path)

In [ ]:

ai_na = usdf[usdf['AI ROLE'].isna()]
print("Number of NAs: ", len(ai_na))
print("loading skill IDs")
# Classify Jobs With AI Skills
with open('../data/ai_skill_ids.pkl', 'rb') as f:
    ai_skill_ids = pickle.load(f)
ai_skill_ids
# Function to check if any AI skill ID is in the job's skills
def has_ai_skills(skill_list, ai_skill_ids):
    skill_list = eval(skill_list)  # Convert string representation of list to actual list
    return any(skill_id in skill_list for skill_id in ai_skill_ids)
ai_na['AI ROLE'] = ai_na.swifter.progress_bar(True).apply(lambda x: has_ai_skills(x['SKILLS'], ai_skill_ids), axis=1)

usdf.loc[ai_na.index, 'AI ROLE'] = ai_na['AI ROLE']



In [ ]:
ai_na = usdf[usdf['AI ROLE'].isna()]
len(ai_na)

In [ ]:
usdf.columns

In [ ]:
# look at skills of an AI ROLE in 2018
for i, row in usdf[(usdf['AI ROLE']==True) & (usdf['YEAR']==2018)].head(10).iterrows():
    print(row['SKILLS_NAME'])

In [ ]:
usdf.to_parquet('../data/us_10m_nointernship_ai_skills_body.parquet.gzip', compression='gzip')

In [ ]:
# export usdf without body
usdf.drop(columns=['BODY']).to_parquet('../data/us_10m_nointernship_ai_skills_benefits.parquet.gzip', compression='gzip')